# TUẦN 4: TINH CHỈNH (FINE-TUNING) VINAI PHOBERT V2 TRÊN GPU
## (STATE-OF-THE-ART TRANSFORMER FOR VIETNAMESE SENTIMENT ANALYSIS)

> **Dự án:** Hệ thống Phân tích Cảm xúc Đánh giá Khách hàng Tiếng Việt (Shopee Reviews)  
> **Học phần:** Artificial Intelligence Capstone 2025 • VNUK, Đại học Đà Nẵng  
> **Assignee:** Từ Văn Huy Hoàng (Issue #2)  
> **Mục tiêu kỹ thuật:**  
> 1. Fine-tuning pre-trained `vinai/phobert-base-v2` từ thư viện Hugging Face `transformers` trên GPU (Google Colab / Kaggle).  
> 2. Thiết lập Optimizer `AdamW` phân tầng: learning rate $2 \times 10^{-5}$ cho backbone và $10^{-3}$ cho classification head.  
> 3. Áp dụng Linear Warmup Scheduler (10% tổng số bước).  
> 4. Tích hợp Weighted Cross-Entropy Loss $w_c = \frac{N}{K \cdot N_c}$ khắc phục mất cân bằng nhãn (RQ3).  
> 5. **Chỉ tiêu nghiệm thu (DoD):** Phá vỡ mốc trần Baseline, đạt **Macro F1 >= 75.0%** trên tập kiểm thử độc lập 5.292 mẫu (hoặc toàn bộ test set).


## 1. KIỂM TRA MÔI TRƯỜNG TÍNH TOÁN GPU & CÀI ĐẶT PHỤ THUỘC

In [ ]:
# Nếu chạy trên Google Colab / Kaggle, bỏ comment dòng dưới:
# !pip install -q transformers pyvi scikit-learn seaborn matplotlib

import os
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Thiết bị sử dụng: {device}")
if torch.cuda.is_available():
    print(f"🔥 GPU Tên: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM Khả dụng: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Lưu ý: Đang chạy trên CPU. Khuyến nghị bật GPU T4/V100/A100 trên Google Colab để fine-tuning nhanh chóng!")


## 2. NẠP DỮ LIỆU & TOKENIZER VINAI PHOBERT V2

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "vinai/phobert-base-v2"
print(f"📥 Đang tải Tokenizer tiền huấn luyện: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

path_data = ROOT_DIR / "data" / "processed"
df_train = pd.read_csv(path_data / "train.csv")
df_val = pd.read_csv(path_data / "val.csv")
df_test = pd.read_csv(path_data / "test.csv")

print(f"✅ Đã nạp Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,} mẫu")


## 3. THIẾT KẾ PYTORCH DATASET CHO TRANSFORMER

In [ ]:
from src.dataset import MAX_SEQUENCE_LENGTH
from src.train import PhoBERTReviewDataset

train_dataset = PhoBERTReviewDataset(
    texts=df_train["processed_content"].tolist(),
    labels=df_train["label_id"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_SEQUENCE_LENGTH,
)
val_dataset = PhoBERTReviewDataset(
    texts=df_val["processed_content"].tolist(),
    labels=df_val["label_id"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_SEQUENCE_LENGTH,
)
test_dataset = PhoBERTReviewDataset(
    texts=df_test["processed_content"].tolist(),
    labels=df_test["label_id"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_SEQUENCE_LENGTH,
)

BATCH_SIZE = 32 if torch.cuda.is_available() else 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Số batch train: {len(train_loader)} (Batch Size = {BATCH_SIZE})")


## 4. KHỞI TẠO KIẾN TRÚC PHOBERTCLASSIFIER & OPTIMIZER PHÂN TẦNG
- Backbone: learning rate = `2e-5`
- Head: learning rate = `1e-3`
- Linear Warmup Scheduler: 10% warmup steps.

In [ ]:
from src.models import (
    PhoBERTClassifier,
    compute_class_weights,
    get_weighted_criterion,
    get_phobert_optimizer_and_scheduler,
)

model_phobert = PhoBERTClassifier(
    model_name=MODEL_NAME,
    num_classes=3,
    dropout_rate=0.2,
).to(device)

# Weighted Cross-Entropy Loss cho RQ3
class_weights = compute_class_weights(df_train["label_id"].values, num_classes=3, device=device)
criterion = get_weighted_criterion(class_weights, device=device)
print(f"⚖️ Class weights (RQ3): {class_weights.cpu().numpy().tolist()}")

EPOCHS = 3
total_steps = len(train_loader) * EPOCHS
optimizer, scheduler = get_phobert_optimizer_and_scheduler(
    model=model_phobert,
    total_steps=total_steps,
    lr_backbone=2e-5,
    lr_head=1e-3,
    warmup_ratio=0.1,
)
print(f"✅ Đã thiết lập AdamW phân tầng & Linear Warmup Scheduler ({int(total_steps*0.1)} warmup steps)")


## 5. HUẤN LUYỆN MÔ HÌNH VỚI CHECKPOINTING

In [ ]:
from src.models import train_one_epoch, evaluate_model, save_checkpoint, load_checkpoint
from src.evaluate import compute_classification_metrics

best_checkpoint_path = ROOT_DIR / "models" / "best_phobert.pt"
best_val_f1 = -1.0

print(f"🚀 BẮT ĐẦU FINE-TUNING PHOBERT V2 TRÊN {device}...\n")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(
        model=model_phobert,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        clip_grad=1.0,
        scheduler=scheduler,
    )
    val_loss, val_acc, y_val_t, y_val_p = evaluate_model(
        model=model_phobert,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )
    val_metrics = compute_classification_metrics(y_val_t, y_val_p)
    val_macro_f1 = float(val_metrics["macro_f1"])
    elapsed = time.time() - t0
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% | Val Macro F1: {val_macro_f1*100:.2f}%")
    
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        save_checkpoint(
            model=model_phobert,
            optimizer=optimizer,
            epoch=epoch,
            metrics={"val_macro_f1": val_macro_f1, "val_acc": val_acc},
            save_path=best_checkpoint_path,
            extra_state={
                "model_type": "phobert",
                "model_name": MODEL_NAME,
                "num_classes": 3,
                "class_weights": class_weights.cpu().tolist(),
            }
        )
        print(f"   --> 🌟 Checkpoint tối ưu mới: Val Macro F1 = {val_macro_f1*100:.2f}%")


## 6. ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP (RQ2 SOTA BENCHMARK)
Mục tiêu nghiệm thu: **Macro F1 >= 75.0%**.

In [ ]:
from src.evaluate import print_metrics_summary, plot_and_save_confusion_matrix, analyze_error_samples

print(f"📥 Nạp checkpoint tối ưu từ: {best_checkpoint_path}")
load_checkpoint(model_phobert, best_checkpoint_path, device=device)

test_loss, test_acc, y_test_t, y_test_p = evaluate_model(
    model=model_phobert,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
)
test_metrics = compute_classification_metrics(y_test_t, y_test_p)
print_metrics_summary(test_metrics)

cm_path = ROOT_DIR / "docs" / "phobert_confusion_matrix.png"
plot_and_save_confusion_matrix(y_test_t, y_test_p, save_path=str(cm_path), title="PhoBERT v2 Confusion Matrix (Normalized)")


## 7. BẢNG TỔNG HỢP HIỆU NĂNG BENCHMARK (RQ1, RQ2, RQ3)

In [ ]:
benchmark_df = pd.DataFrame({
    "Kiến trúc Mô hình": [
        "1. Baseline: Logistic Regression (TF-IDF)",
        "2. Baseline: LinearSVC (TF-IDF)",
        "3. Deep Learning: Bi-LSTM (Weighted Loss)",
        "4. SOTA: VinAI PhoBERT v2 (Fine-tuned)",
    ],
    "Accuracy": ["74.80%", "75.90%", "74.19%", f"{test_metrics['accuracy']*100:.2f}%"],
    "Macro F1": ["64.20%", "65.99%", "66.27%", f"{test_metrics['macro_f1']*100:.2f}%"],
    "F1 (NEG)": ["68.10%", "69.50%", "70.10%", f"{test_metrics['per_class']['NEG']*100:.2f}%"],
    "F1 (NEU)": ["38.50%", "42.80%", "42.22%", f"{test_metrics['per_class']['NEU']*100:.2f}%"],
    "F1 (POS)": ["86.00%", "86.60%", "86.47%", f"{test_metrics['per_class']['POS']*100:.2f}%"],
    "Vượt Baseline F1": ["-", "Mốc chuẩn", "+0.28%", f"+{test_metrics['macro_f1']*100 - 65.99:.2f}%"]
})

print("🏆 BẢNG TỔNG HỢP HIỆU NĂNG TOÀN DỰ ÁN:")
display(benchmark_df) if "display" in dir() else print(benchmark_df.to_string(index=False))


## 8. KIỂM THỬ SUY DIỄN THỜI GIAN THỰC CHO WEB DEMO (APP.PY)
Tiêu chuẩn: **Thời gian suy diễn dưới 100ms/câu**.

In [ ]:
from src.inference import predict_sentiment

demo_inputs = [
    "Giao hàng siêu nhanh, đóng gói kỹ càng, chất lượng chuẩn auth 5 sao!",
    "Hàng dỏm, đường may rách nát, lừa đảo khách hàng.",
    "Màu sắc tạm ổn, vải hơi mỏng, tiền nào của nấy thôi.",
    "",  # Kiểm thử câu rỗng
]

print("⚡ KẾT QUẢ SUY DIỄN PHOBERT:")
for inp in demo_inputs:
    res = predict_sentiment(inp, model_type="phobert")
    print(f"\nReview: \"{inp}\"")
    print(f" -> Nhãn: {res['label']} (Confidence: {res['confidence']*100:.1f}%) | Latency: {res['latency_ms']:.2f}ms")
    print(f" -> Xác suất: {res['probabilities']}")
